# 🤖 Inference Comparison: Base vs DAPT Fine-Tuned Model
This notebook compares the outputs of the base DeepSeek model and your domain-adapted version using real university-related questions.

# # 🤖 RAG Inference Comparison
# This notebook compares the outputs of multiple models when used
# as the generator within a Retrieval-Augmented Generation (RAG) pipeline.
# The retriever uses the FAISS index built from 'health_and_safety.txt'.
# Models compared:
# 1. Base DeepSeek 8B model
# 2. DAPT 8B model (QLoRA Adapters)
# 3. SFT DAPT 8B model (QLoRA Adapters)
# 4. Base DeepSeek Coder 1.3B model
# 5. DAPT 1.3B model (FP32)
# 6. SFT DAPT 1.3B model (FP32)
# Models are loaded sequentially to conserve VRAM.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from peft import PeftModel
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
from langchain.schema.output_parser import StrOutputParser
from langchain_huggingface import HuggingFacePipeline 
from langchain_core.documents import Document 
import pandas as pd
import torch
import gc
import os
import time 
from typing import List, Dict, Any 

# === CONFIG ===
VECTORSTORE_PATH = "faiss_health_safety_index_bge" 
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"
NUM_CHUNKS_TO_RETRIEVE = 3 

# --- Model Paths ---
BASE_MODEL_8B = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
BASE_MODEL_1_3B = "deepseek-ai/deepseek-coder-1.3b-base"
DAPT_ADAPTER_PATH_8B = "../1.dapt/dapt_qlora_model_output"
DAPT_MODEL_PATH_1_3B = "../1.dapt/dapt_deepseek_1.3b_model_output" 
SFT_DAPT_ADAPTER_PATH_8B = "../2.sft/sft_dapt_8b_qlora_output"
SFT_DAPT_MODEL_PATH_1_3B = "../2.sft/sft_dapt_1.3b_fp32_output"   

# --- Other Files ---
QUESTIONS_FILE = "../healthandsafety_eval_questions_extended.txt" 
OUTPUT_CSV_FILE = "rag_model_comparison_output.csv"

# --- Hardware & Inference Settings ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LOAD_IN_4BIT_INFERENCE_8B = True
compute_dtype_4bit = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
dtype_1_3b = torch.float32 

# --- Generation Parameters ---
MAX_NEW_TOKENS = 250 


In [ ]:
# --- Helper Functions ---
def clear_memory():
    """Clears GPU cache and runs Python garbage collector."""
    print("🧹 Clearing CUDA cache and collecting garbage...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    time.sleep(1)
    print("✅ Memory cleared.")

def format_docs(docs: List[Document]) -> str:
    """Concatenates page content of retrieved documents."""
    return "\n\n".join(doc.page_content for doc in docs)

# --- Configure Quantization for 8B ---
bnb_config_inference_8b = None
if LOAD_IN_4BIT_INFERENCE_8B:
    print(f"⚙️ Configuring 4-bit quantization for 8B inference (compute dtype: {compute_dtype_4bit})...")
    bnb_config_inference_8b = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype_4bit,
    )

# --- Check if paths exist ---
model_paths_to_check = {
    'dapt_8b_adapter': DAPT_ADAPTER_PATH_8B,
    'sft_dapt_8b_adapter': SFT_DAPT_ADAPTER_PATH_8B,
    'dapt_1.3b_model': DAPT_MODEL_PATH_1_3B,
    'sft_dapt_1.3b_model': SFT_DAPT_MODEL_PATH_1_3B,
    'vectorstore': VECTORSTORE_PATH
}
model_paths_exist = {key: os.path.exists(path) for key, path in model_paths_to_check.items()}
for key, exists in model_paths_exist.items():
    if not exists:
         print(f"⚠️ WARNING: Path for '{key}' not found: {model_paths_to_check[key]}")


In [ ]:
# --- Load Retriever ---
retriever = None
if model_paths_exist['vectorstore']:
    print(f"\n--- Loading Retriever ---")
    try:
        print(f"🧠 Loading embedding model: {EMBEDDING_MODEL_NAME}")
        embeddings = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL_NAME,
            model_kwargs={'device': DEVICE},
            encode_kwargs={'normalize_embeddings': True}
        )
        print(f"💾 Loading FAISS index from: {VECTORSTORE_PATH}")
        vectorstore = FAISS.load_local(
            VECTORSTORE_PATH,
            embeddings,
            allow_dangerous_deserialization=True
        )
        retriever = vectorstore.as_retriever(search_kwargs={"k": NUM_CHUNKS_TO_RETRIEVE})
        print(f"✅ Retriever initialized to fetch {NUM_CHUNKS_TO_RETRIEVE} chunks.")
    except Exception as e:
        print(f"❌ Error loading retriever: {e}")
        retriever = None
else:
    print(f"❌ Cannot proceed with RAG: Vector store not found at {VECTORSTORE_PATH}")


In [ ]:
# Load evaluation questions
try:
    questions_path = QUESTIONS_FILE
    if not os.path.exists(questions_path):
         questions_path = os.path.basename(QUESTIONS_FILE)
         if not os.path.exists(questions_path):
             raise FileNotFoundError(f"Questions file not found: {QUESTIONS_FILE}")

    with open(questions_path, "r", encoding="utf-8") as f:
        questions = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(questions)} questions from {questions_path}")
    if not questions: raise ValueError("Questions file is empty.")
    print("First 3 questions:", questions[:3])
    print("Last 3 questions:", questions[-3:])
except Exception as e:
    print(f"❌ Error reading questions file: {e}")
    questions = []


In [ ]:
# Define all models to evaluate with RAG
models_to_evaluate = [
    {"key": "base_8b",         "model_path": BASE_MODEL_8B,              "adapter_path": None,                         "is_peft": False, "quant_config": bnb_config_inference_8b, "dtype": "auto",       "column_name": "Base (8B) + RAG"},
    {"key": "dapt_8b_adapter", "model_path": BASE_MODEL_8B,              "adapter_path": DAPT_ADAPTER_PATH_8B,         "is_peft": True,  "quant_config": bnb_config_inference_8b, "dtype": "auto",       "column_name": "DAPT (8B QLoRA) + RAG"},
    {"key": "sft_dapt_8b_adapter","model_path": BASE_MODEL_8B,            "adapter_path": SFT_DAPT_ADAPTER_PATH_8B,     "is_peft": True,  "quant_config": bnb_config_inference_8b, "dtype": "auto",       "column_name": "SFT DAPT (8B QLoRA) + RAG"},
    {"key": "base_1.3b",       "model_path": BASE_MODEL_1_3B,            "adapter_path": None,                         "is_peft": False, "quant_config": None,                    "dtype": dtype_1_3b, "column_name": "Base (1.3B FP32) + RAG"},
    {"key": "dapt_1.3b_model", "model_path": DAPT_MODEL_PATH_1_3B,       "adapter_path": None,                         "is_peft": False, "quant_config": None,                    "dtype": dtype_1_3b, "column_name": "DAPT (1.3B FP32) + RAG"},
    {"key": "sft_dapt_1.3b_model","model_path": SFT_DAPT_MODEL_PATH_1_3B,"adapter_path": None,                         "is_peft": False, "quant_config": None,                    "dtype": dtype_1_3b, "column_name": "SFT DAPT (1.3B FP32) + RAG"},
]

# Dictionary to store results for each model
all_results = {}

# --- RAG Prompt Template ---
rag_prompt_template = """
You are a helpful assistant representing the University of Bradford staff.
Answer the following question based ONLY on the context provided below.
If the context does not contain the answer, state clearly that you cannot answer based on the provided information or say 'I cannot answer this question based on the provided context.'.
Do not add information that is not present in the context. Be concise and direct.

Context:
{context}

Question:
{question}

Answer:
"""
RAG_PROMPT = PromptTemplate.from_template(rag_prompt_template)

# --- Inference Loop (Model by Model with RAG) ---
print("🚀 Starting RAG Inference Comparison (one model at a time)...")

if not questions:
    print("⚠️ No questions loaded, skipping inference loop.")
elif retriever is None:
    print("⚠️ Retriever not loaded, skipping RAG inference loop.")
else:
    # --- Loop through each model configuration ---
    for model_config in models_to_evaluate:
        model_key = model_config["key"]
        adapter_path = model_config["adapter_path"]
        column_name = model_config["column_name"]
        is_peft = model_config["is_peft"]
        model_path = model_config["model_path"]
        quant_config = model_config["quant_config"]
        load_dtype = model_config["dtype"]

        # Skip if required paths don't exist for non-base models
        if model_key not in ["base_8b", "base_1.3b"] and not model_paths_exist[model_key]:
             print(f"\n--- Skipping model '{column_name}' (Required path not found) ---")
             all_results[column_name] = ["Model/Adapters not found"] * len(questions)
             continue

        print(f"\n--- Clearing memory before loading {column_name} ---")
        clear_memory() # Aggressive clearing before load attempt

        print(f"\n--- Loading Generator LLM: {column_name} ---")
        llm = None
        tokenizer = None
        model_answers = [] # Store answers for this specific model

        try:
            # Load the tokenizer
            base_for_tokenizer = BASE_MODEL_8B if "8b" in model_key else BASE_MODEL_1_3B
            tokenizer_path = base_for_tokenizer if is_peft else model_path
            print(f"Loading tokenizer from: {tokenizer_path}")
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
            if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

            # Load the base or full model
            print(f"Loading model from: {model_path} with dtype: {load_dtype}")
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                quantization_config=quant_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=load_dtype
            )

            # Apply adapters if needed
            if is_peft:
                print(f"Applying adapters from {adapter_path}...")
                model = PeftModel.from_pretrained(model, adapter_path)
                print("Adapters applied.")

            # Create HF Pipeline for Langchain integration
            print(f"Creating text generation pipeline for {column_name}...")
            llm_pipeline = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
           
            llm = HuggingFacePipeline(pipeline=llm_pipeline)
            print(f"✅ Generator LLM {column_name} loaded successfully.")

     
            rag_chain = (
        
                {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
                | RAG_PROMPT # Format the prompt
                | llm        # Call the loaded LLM
                | StrOutputParser() # Get the string output
            )
            print("✅ RAG chain created for current LLM.")


            # --- Generate answers for all questions with this RAG chain ---
            for i, q in enumerate(questions):
                print(f"  Processing question {i+1}/{len(questions)} for {column_name} + RAG...")
                try:
                    # Invoke the RAG chain with the question
                    answer = rag_chain.invoke(q)
                    model_answers.append(answer.strip())

                except Exception as e_gen:
                    print(f"  ❌ Error during RAG generation for question {i+1} with {column_name}: {e_gen}")
                    model_answers.append(f"ERROR: RAG Generation failed")

            all_results[column_name] = model_answers

        except Exception as e_load:
            print(f"❌ Error loading or processing model {column_name}: {e_load}")
            import traceback
            traceback.print_exc()
            all_results[column_name] = [f"ERROR loading model"] * len(questions)
        finally:
            # --- Crucially: Delete model and pipeline, clear memory ---
            print(f"--- Unloading Model: {column_name} ---")
            if 'llm_pipeline' in locals() and llm_pipeline is not None: del llm_pipeline
            if 'llm' in locals() and llm is not None: del llm
            if 'model' in locals() and model is not None: del model
            if 'tokenizer' in locals() and tokenizer is not None: del tokenizer
            clear_memory()
            print(f"✅ Model {column_name} unloaded.")

    print("\n✅ All models processed with RAG.")

# --- Combine Results into DataFrame ---
if questions:
    df_data = {"Question": questions}
    for model_config in models_to_evaluate:
        col_name = model_config["column_name"]
        if col_name in all_results:
             if len(all_results[col_name]) == len(questions):
                 df_data[col_name] = all_results[col_name]
             else:
                 print(f"⚠️ Warning: Mismatch in number of answers for {col_name}. Expected {len(questions)}, got {len(all_results[col_name])}. Filling with errors.")
                 df_data[col_name] = ["ERROR: Answer list length mismatch"] * len(questions)
        else:
             df_data[col_name] = ["Model failed to load/run"] * len(questions)

    df = pd.DataFrame(df_data)

    # Define desired column order including all RAG models
    column_order = [
        "Question",
        "Base (8B) + RAG",
        "DAPT (8B QLoRA) + RAG",
        "SFT DAPT (8B QLoRA) + RAG",
        "Base (1.3B FP32) + RAG",
        "DAPT (1.3B FP32) + RAG",
        "SFT DAPT (1.3B FP32) + RAG"
    ]
    # Filter for columns that actually have results or error messages
    existing_columns = [col for col in column_order if col in df.columns]
    if existing_columns:
        df = df[existing_columns]
        print("\n--- RAG Comparison Results ---")
        pd.set_option('display.max_colwidth', 150) # Show more content per cell
        pd.set_option('display.width', 1000)       # Increase display width
        print(df.to_markdown(index=False))
    else:
        print("\n⚠️ No results generated or models loaded successfully.")
else:
    print("\n⚠️ No questions loaded, cannot generate results DataFrame.")
    df = pd.DataFrame()

In [ ]:
# Save to CSV
if not df.empty:
    try:
        df.to_csv(OUTPUT_CSV_FILE, index=False)
        print(f"\n✅ RAG comparison results saved to {OUTPUT_CSV_FILE}")
    except Exception as e:
        print(f"❌ Error saving results to CSV: {e}")
else:
    print("\n⚠️ DataFrame is empty, skipping CSV save.")
